## 1. Construção da Camada Silver: `tb_info_filmes`

Nesta etapa, processamos a tabela bruta **`cinedata_bronze.tb_movies_info`** para gerar a tabela enriquecida **`cinedata_silver.tb_info_filmes`**.

---

### 📌 Tratamentos Aplicados:
- **Mapeamento e Tradução:** Renomeação de todas as colunas para o português.
- **Tratamento de Datas:** Conversão robusta aceitando múltiplos formatos de entrada (`yyyy-MM-dd`, `dd/MM/yyyy`, etc.), atribuindo `NULL` apenas em casos irrecuperáveis.
- **Coluna Derivada:** Extração da coluna `ano_lancamento` a partir da `data_lancamento`.
- **Normalização e De-Para do Status:** Remoção de caracteres especiais/hífens, padronização de caixa e tradução para o português (`Lançado`, `Pós-Produção`, etc.), tratando registros inválidos como `"Não Informado"`.
- **Deduplicação:** Aplicação de *Window Function* para manter a versão mais recente (`ingestion_datetime`) de cada `id_filme`.
- **Persistência:** Gravado na camada Silver em formato Delta Lake.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, coalesce, try_to_date, year, 
    when, row_number, current_timestamp, expr
)

# 1. Definição dos Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura da Tabela Bronze
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# 3. Tratamento e Parsing de Datas Multi-Formato usando try_to_date
df_info_parsed = df_bronze_info.withColumn(
    "data_lancamento_parsed",
    coalesce(
        try_to_date(col("release_date"), "yyyy-MM-dd"),
        try_to_date(col("release_date"), "dd/MM/yyyy"),
        try_to_date(col("release_date"), "MM/dd/yyyy"),
        try_to_date(col("release_date"), "MM-dd-yyyy"),
        try_to_date(col("release_date"), "yyyy/MM/dd")
    )
)

# 4. Normalização e Tradução do Status
df_info_cleaned = df_info_parsed.withColumn(
    "status_clean",
    trim(regexp_replace(lower(col("status")), "[-_]", " "))
)

df_info_status = df_info_cleaned.withColumn(
    "status_filme_traduzido",
    when(col("status_clean") == "released", "Lançado")
    .when(col("status_clean") == "post production", "Pós-Produção")
    .when(col("status_clean") == "in production", "Em Produção")
    .when(col("status_clean") == "planned", "Planejado")
    .when(col("status_clean") == "rumored", "Rumores")
    .when(col("status_clean") == "canceled", "Cancelado")
    .when(col("status_clean") == "cancelled", "Cancelado")
    .otherwise("Não Informado")
)

# 5. Mapeamento, Seleção e Tipagem Segura com try_cast
df_info_mapped = df_info_status.select(
    expr("try_cast(id as long)").alias("id_filme"),
    col("title").alias("titulo"),
    col("original_title").alias("titulo_original"),
    col("data_lancamento_parsed").alias("data_lancamento"),
    year(col("data_lancamento_parsed")).cast("integer").alias("ano_lancamento"),
    expr("try_cast(runtime as integer)").alias("duracao_minutos"),
    col("original_language").alias("idioma_original"),
    col("status_filme_traduzido").alias("status_filme"),
    col("overview").alias("sinopse"),
    col("ingestion_datetime")
).filter(col("id_filme").isNotNull()) # Filtra linhas ruidosas/corrompidas onde o ID não era um número

# 6. Deduplicação por Filme (mantendo a versão mais recente por ingestion_datetime)
window_spec = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())

df_info_dedup = df_info_mapped \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "ingestion_datetime") \
    .withColumn("processing_datetime", current_timestamp())

# 7. Gravação na Camada Silver (Delta Lake)
df_info_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

# 8. Validação e Exibição do Resultado
print(f"[OK] Tabela 'tb_info_filmes' gravada com sucesso em {silver_schema}!")
display(spark.table(f"{silver_schema}.tb_info_filmes").limit(5))

[OK] Tabela 'tb_info_filmes' gravada com sucesso em workspace.cinedata_silver!


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,processing_datetime
38258,Grizzly II: Revenge,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly,2026-09-21T12:37:46.213Z
38700,Bad Boys for Life,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",2026-09-21T12:37:46.213Z
45033,20 Seconds of Joy,20 Seconds of Joy,2018-01-01,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",2026-09-21T12:37:46.213Z
46983,The Song of Styrene,Le Chant du styrène,2022-05-23,2022,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,2026-09-21T12:37:46.213Z
66534,Annanukku Jey,அண்ணனுக்கு ஜே,2018-08-31,2018,111,ta,Lançado,The son of a toddy seller decides to become a politician to take on bar owner who tries to bring down his father.,2026-09-21T12:37:46.213Z



## 2. Construção da Camada Silver: `tb_financeiro_filmes`

Nesta etapa, processamos a tabela bruta **`cinedata_bronze.tb_movies_financials`** para gerar a tabela financeira **`cinedata_silver.tb_financeiro_filmes`**.

---

### 📌 Tratamentos Aplicados:
- **Higienização de Texto/Moeda:** Remoção de caracteres de moedas (`$`, `R$`), pontuações de milhar e strings de dados ausentes (`"Unknown"`, `"Não Informado"`), convertendo-os em `NULL`.
- **Validação de Métricas Numericas:** Conversão para o tipo `DECIMAL(18,2)` e tratamento para que valores zerados (`<= 0`) sejam convertidos em `NULL`.
- **Conversão Financeira (USD -> BRL):** Aplicação da cotação do dólar obtida da `cinedata_bronze.tb_cotacao_dolar` para converter os valores de orçamento e receita para Reais (BRL).
- **Métricas Derivadas Seguras:**
  - `lucro_usd` = `receita_usd - orcamento_usd`
  - `lucro_brl` = `receita_brl - orcamento_brl`
  - `margem_lucro_pct` = `(lucro_usd / receita_usd) * 100`, tratada com `when` para evitar divisão por zero.
- **Deduplicação e Persistência:** Deduplicação por `id_filme` retendo o registro mais recente via `ingestion_datetime` e gravação no formato Delta Lake.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace,
    coalesce,
    when,
    row_number,
    current_timestamp,
    expr,
)

# 1. Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura das Tabelas da Bronze
df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# 3. Obtenção da taxa de cotação mais recente do Dólar (BRL)
taxa_dolar_row = (
    df_bronze_cotacao.orderBy(
        col("ingestion_datetime").desc(), col("dataHoraCotacao").desc()
    )
    .select("cotacaoCompra")
    .first()
)

taxa_dolar = (
    float(taxa_dolar_row["cotacaoCompra"])
    if taxa_dolar_row and taxa_dolar_row["cotacaoCompra"]
    else 5.25
)
print(f"[INFO] Taxa de Cotação do Dólar utilizada: R$ {taxa_dolar:.2f}")


# 4. Higienização das Colunas de Orçamento e Receita
def higienizar_metrica(coluna_str):
  # Limpa caracteres de moeda e caracteres especiais
  limpo = trim(regexp_replace(lower(col(coluna_str)), r"[\$\,\s]|r\$", ""))
  # Transforma textos de dados ausentes em NULL
  limpo_nulls = when(
      limpo.isin("unknown", "não informado", "nao informado", "none", "null", ""),
      None,
  ).otherwise(limpo)
  # Usa try_cast via expr do Spark SQL para não depender de importação do Python
  num_decimal = expr("try_cast(" + coluna_str + " as decimal(18,2))")
  return when(num_decimal > 0, num_decimal).otherwise(None)


df_fin_clean = (
    df_bronze_financials.withColumn("id_filme", expr("try_cast(id as long)"))
    .withColumn("orcamento_usd", higienizar_metrica("budget"))
    .withColumn("receita_usd", higienizar_metrica("revenue"))
    .filter(col("id_filme").isNotNull())
)

# 5. Cálculo dos Valores em Reais (BRL)
df_fin_brl = df_fin_clean.withColumn(
    "orcamento_brl", (col("orcamento_usd") * taxa_dolar).cast("decimal(18,2)")
).withColumn(
    "receita_brl", (col("receita_usd") * taxa_dolar).cast("decimal(18,2)")
)

# 6. Cálculo Seguro de Lucro e Margem de Lucro Percentual
df_fin_metrics = (
    df_fin_brl.withColumn(
        "lucro_usd",
        (col("receita_usd") - col("orcamento_usd")).cast("decimal(18,2)"),
    )
    .withColumn(
        "lucro_brl",
        (col("receita_brl") - col("orcamento_brl")).cast("decimal(18,2)"),
    )
    .withColumn(
        "margem_lucro_pct",
        when(
            (col("receita_usd").isNotNull())
            & (col("receita_usd") > 0)
            & (col("lucro_usd").isNotNull()),
            ((col("lucro_usd") / col("receita_usd")) * 100).cast(
                "decimal(10,2)"
            ),
        ).otherwise(None),
    )
)

# 7. Deduplicação por Filme (versão mais recente por ingestion_datetime)
window_spec = Window.partitionBy("id_filme").orderBy(
    col("ingestion_datetime").desc()
)

df_fin_dedup = (
    df_fin_metrics.withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_pct",
    )
    .withColumn("processing_datetime", current_timestamp())
)

# 8. Gravação na Camada Silver (Delta Lake)
df_fin_dedup.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

# 9. Validação e Exibição do Resultado
print(
    f"[OK] Tabela 'tb_financeiro_filmes' gravada com sucesso em {silver_schema}!"
)
display(spark.table(f"{silver_schema}.tb_financeiro_filmes").limit(5))

[INFO] Taxa de Cotação do Dólar utilizada: R$ 5.16
[OK] Tabela 'tb_financeiro_filmes' gravada com sucesso em workspace.cinedata_silver!


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct,processing_datetime
14564,25000000.00,83080890.00,128922500.00,428439841.64,58080890.00,299517341.64,69.91,2026-09-21T12:38:02.366Z
32471,null,null,null,null,null,null,null,2026-09-21T12:38:02.366Z
38258,null,null,null,null,null,null,null,2026-09-21T12:38:02.366Z
38492,null,null,null,null,null,null,null,2026-09-21T12:38:02.366Z
38700,null,426505244.00,null,2199444892.78,null,null,null,2026-09-21T12:38:02.366Z



## 3. Construção da Camada Silver: `tb_metricas_engajamento`

Nesta etapa, processamos a tabela bruta **`cinedata_bronze.tb_movies_metrics`** para gerar a tabela **`cinedata_silver.tb_metricas_engajamento`**.

---

### 📌 Tratamentos Aplicados:
- **Tratamento de Separadores Decimais:** Higienização da coluna `popularity`, substituindo vírgulas por pontos antes do cast numérico.
- **Tratamento de Column Shift:** Aplicação de `try_cast` via `expr` para converter com segurança textos deslocados nas colunas de notas e votos em `NULL`.
- **Limites de Escala de Negócio:**
  - `nota_media_tmdb` e `nota_media_imdb`: Validação de intervalo válido entre **0 e 10**. Valores fora desse limite viram `NULL`.
  - `popularity`, `qtd_votos_tmdb` e `qtd_votos_imdb`: Invalidação e conversão de valores **negativos** (`< 0`) para `NULL`.
- **Deduplicação e Persistência:** Aplicação de *Window Function* para reter apenas a última versão de cada `id_filme` e gravação em Delta Lake no schema Silver.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, when, row_number, current_timestamp, expr
)

# 1. Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura da Tabela Bronze
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

# 3. Limpeza Inicial de Strings e Conversão Segura (Column Shift Safe)
df_metrics_cleaned = df_bronze_metrics.select(
    expr("try_cast(id as long)").alias("id_filme"),
    
    # Popularidade: Limpa espaços e ajusta vírgula/ponto antes da conversão
    expr("try_cast(regexp_replace(trim(popularity), ',', '.') as decimal(10,4))").alias("popularidade_raw"),
    
    # Notas Média (TMDB e IMDb): Cast seguro
    expr("try_cast(trim(vote_average) as decimal(4,2))").alias("nota_tmdb_raw"),
    expr("try_cast(trim(averageRating) as decimal(4,2))").alias("nota_imdb_raw"),
    
    # Quantidade de Votos: Cast seguro para inteiro/long
    expr("try_cast(trim(vote_count) as long)").alias("votos_tmdb_raw"),
    expr("try_cast(trim(numVotes) as long)").alias("votos_imdb_raw"),
    
    col("ingestion_datetime")
).filter(col("id_filme").isNotNull())

# 4. Aplicação das Regras de Negócio e Limites de Escala
df_metrics_validated = df_metrics_cleaned \
    .withColumn(
        "popularidade",
        when(col("popularidade_raw") >= 0, col("popularidade_raw")).otherwise(None)
    ) \
    .withColumn(
        "nota_media_tmdb",
        when((col("nota_tmdb_raw") >= 0) & (col("nota_tmdb_raw") <= 10), col("nota_tmdb_raw")).otherwise(None)
    ) \
    .withColumn(
        "qtd_votos_tmdb",
        when(col("votos_tmdb_raw") >= 0, col("votos_tmdb_raw")).otherwise(None)
    ) \
    .withColumn(
        "nota_media_imdb",
        when((col("nota_imdb_raw") >= 0) & (col("nota_imdb_raw") <= 10), col("nota_imdb_raw")).otherwise(None)
    ) \
    .withColumn(
        "qtd_votos_imdb",
        when(col("votos_imdb_raw") >= 0, col("votos_imdb_raw")).otherwise(None)
    )

# 5. Deduplicação por Filme (mantendo a versão mais recente por ingestion_datetime)
window_spec = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())

df_metrics_dedup = df_metrics_validated \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .select(
        "id_filme",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    ) \
    .withColumn("processing_datetime", current_timestamp())

# 6. Gravação na Camada Silver (Delta Lake)
df_metrics_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

# 7. Validação e Exibição do Resultado
print(f"[OK] Tabela 'tb_metricas_engajamento' gravada com sucesso em {silver_schema}!")
display(spark.table(f"{silver_schema}.tb_metricas_engajamento").limit(5))

[OK] Tabela 'tb_metricas_engajamento' gravada com sucesso em workspace.cinedata_silver!


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,processing_datetime
14564,24.5840,4.97,2375,null,46286,2026-09-21T12:38:08.171Z
32471,8.9290,7.06,118,6.60,4617,2026-09-21T12:38:08.171Z
38258,null,3.16,28,null,null,2026-09-21T12:38:08.171Z
38492,2.9800,7.10,null,7.80,212,2026-09-21T12:38:08.171Z
38700,46.6190,7.14,7570,6.50,199420,2026-09-21T12:38:08.171Z


## 4. Construção da Camada Silver: `tb_avaliacoes_usuarios`

Nesta etapa, processamos a tabela bruta **`cinedata_bronze.tb_movies_reviews`** para criar a tabela **`cinedata_silver.tb_avaliacoes_usuarios`**.

---

### 📌 Tratamentos Aplicados:
- **Mapeamento e Tipagem:** Renomeação das colunas para o português e cast seguro de `id_filme` para `LONG` e `nota_usuario` para `DECIMAL(4,2)`.
- **Validação da Escala de Notas:** Aplicação de regra de negócio para que notas fora do intervalo [0, 10] sejam convertidas para `NULL`.
- **Tratamento de Comentários:** Identificação de comentários nulos, vazios ou compostos apenas por espaços, substituindo-os pelo texto padronizado `"Sem comentário"`.
- **Deduplicação Completa:** Remoção de duplicatas exatas pela combinação de filme, usuário, nota e comentário via `dropDuplicates()`.
- **Persistência:** Salvamento na camada Silver em formato Delta Lake.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, when, current_timestamp, expr
)

# 1. Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura da Tabela Bronze
df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

# 3. Mapeamento, Limpeza Inicial e Tipagem Segura
df_reviews_mapped = df_bronze_reviews.select(
    expr("try_cast(id as long)").alias("id_filme"),
    trim(col("nome")).alias("nome_usuario"),
    expr("try_cast(trim(nota) as decimal(4,2))").alias("nota_raw"),
    trim(col("comentario")).alias("comentario_raw")
).filter(col("id_filme").isNotNull())

# 4. Tratamento de Escala de Nota (0 a 10) e Padronização de Comentários
df_reviews_treated = df_reviews_mapped \
    .withColumn(
        "nota_usuario",
        when((col("nota_raw") >= 0) & (col("nota_raw") <= 10), col("nota_raw")).otherwise(None)
    ) \
    .withColumn(
        "comentario_usuario",
        when(
            (col("comentario_raw").isNull()) | (col("comentario_raw") == ""),
            "Sem comentário"
        ).otherwise(col("comentario_raw"))
    ) \
    .select(
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    )

# 5. Remoção de Registros Integralmente Duplicados
df_reviews_dedup = df_reviews_treated \
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]) \
    .withColumn("processing_datetime", current_timestamp())

# 6. Gravação na Camada Silver (Delta Lake)
df_reviews_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

# 7. Validação e Exibição do Resultado
print(f"[OK] Tabela 'tb_avaliacoes_usuarios' gravada com sucesso em {silver_schema}!")
display(spark.table(f"{silver_schema}.tb_avaliacoes_usuarios").limit(5))

[OK] Tabela 'tb_avaliacoes_usuarios' gravada com sucesso em workspace.cinedata_silver!


id_filme,nome_usuario,nota_usuario,comentario_usuario,processing_datetime
846452,Sérgio Freitas,6.10,Assisti até o final mas não me marcou.,2026-09-21T12:38:13.596Z
787495,Marcio Lima 164,0.20,Sem comentário,2026-09-21T12:38:13.596Z
476217,Amanda Correia 382,4.90,"Fraco, não recomendo.",2026-09-21T12:38:13.596Z
831775,Juliana Gomes 626,8.00,Muito bom! Vale a pena assistir.,2026-09-21T12:38:13.596Z
857553,Marcelo Castro,9.10,"Obra-prima do cinema, simplesmente espetacular.",2026-09-21T12:38:13.596Z



## 5. Construção da Camada Silver: `tb_generos`

Nesta etapa, processamos a tabela bruta **`cinedata_bronze.tb_credits_and_tags`** para gerar a tabela normalizada **`cinedata_silver.tb_generos`**.

---

### 📌 Tratamentos Aplicados:
- **Padronização de Separadores:** Substituição de `;` por `,` na coluna `genres`.
- **Desmembramento (Explode):** Separação dos gêneros múltiplos em linhas individuais por filme.
- **Higienização de Dados (Column Shift e Sujeiras):**
  - Remoção de espaços extras nas extremidades com `trim()`.
  - Invalidação de gêneros nulos, vazios ou compostos por números/caracteres fora do domínio válido de gêneros.
- **Deduplicação:** Remoção de duplicatas por `(id_filme, nome_genero)` via `dropDuplicates()`.
- **Persistência:** Salvamento no Delta Lake no schema Silver.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, initcap, regexp_replace, split, explode, current_timestamp, expr
)

# 1. Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura da Tabela Bronze
df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# 3. Tratamento de Separadores, Split e Explode
df_generos_exploded = df_bronze_credits.select(
    expr("try_cast(id as long)").alias("id_filme"),
    explode(
        split(
            regexp_replace(col("genres"), ";", ","), 
            ","
        )
    ).alias("genero_raw")
).filter(col("id_filme").isNotNull())

# 4. Limpeza, Capitalização e Filtragem de Anomalias/Column Shift
df_generos_cleaned = df_generos_exploded \
    .withColumn("nome_genero", trim(col("genero_raw"))) \
    .filter(
        (col("nome_genero").isNotNull()) & 
        (col("nome_genero") != "") & 
        # Remove valores numéricos isolados ou sujeiras de Column Shift (deve possuir ao menos 2 letras)
        (col("nome_genero").rlike("(?i)^[a-zà-ú\\s\\-]+$")) &
        (~col("nome_genero").rlike("^\\d+$"))
    ) \
    .withColumn("nome_genero", initcap(col("nome_genero")))

# 5. Deduplicação por Filme e Gênero
df_generos_dedup = df_generos_cleaned \
    .select("id_filme", "nome_genero") \
    .dropDuplicates(["id_filme", "nome_genero"]) \
    .withColumn("processing_datetime", current_timestamp())

# 6. Gravação na Camada Silver (Delta Lake)
df_generos_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

# 7. Validação e Exibição do Resultado
print(f"[OK] Tabela 'tb_generos' gravada com sucesso em {silver_schema}!")
display(spark.table(f"{silver_schema}.tb_generos").limit(10))

[OK] Tabela 'tb_generos' gravada com sucesso em workspace.cinedata_silver!


id_filme,nome_genero,processing_datetime
346364,Fantasy,2026-09-21T12:38:18.949Z
335797,Music,2026-09-21T12:38:18.949Z
337401,Fantasy,2026-09-21T12:38:18.949Z
262504,Mystery,2026-09-21T12:38:18.949Z
766507,Thriller,2026-09-21T12:38:18.949Z
507086,Action,2026-09-21T12:38:18.949Z
375262,Thriller,2026-09-21T12:38:18.949Z
353491,Science Fiction,2026-09-21T12:38:18.949Z
603692,Crime,2026-09-21T12:38:18.949Z
569547,Science Fiction,2026-09-21T12:38:18.949Z



## 6. Construção da Camada Silver: `tb_pessoas_empresas`

Nesta etapa, processamos a tabela **`cinedata_bronze.tb_credits_and_tags`** para criar a dimensão unificada **`cinedata_silver.tb_pessoas_empresas`**.

---

### 📌 Tratamentos Aplicados:
- **Consolidação Unificada:** Unificação das origens (`cast`, `directors`, `writers`, `production_companies`) categorizando cada uma com o respetivo `tipo_entidade` (`Ator`, `Diretor`, `Roteirista`, `Produtora`).
- **Desmembramento (Explode):** Aplicação de `split` + `explode` para separar entidades delimitadas por vírgulas ou pontos e vírgulas.
- **Padronização e Limpeza:**
  - Aplicação de `initcap()` para padronizar a capitalização dos nomes.
  - Invalidação de registos nulos, vazios ou caracteres numéricos residuais decorrentes de *Column Shift*.
- **Deduplicação e Persistência:** Remoção de duplicatas por `(id_filme, nome_entidade, tipo_entidade)` e gravação em formato Delta Lake.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, initcap, regexp_replace, split, explode, lit, current_timestamp, expr
)

# 1. Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura da Tabela Bronze
df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Função auxiliar para processar, aplicar split, explode e mapear o tipo de entidade
def process_entity_column(df, source_col, entity_type):
    return df.select(
        expr("try_cast(id as long)").alias("id_filme"),
        explode(
            split(
                regexp_replace(col(source_col), ";", ","),
                ","
            )
        ).alias("nome_raw"),
        lit(entity_type).alias("tipo_entidade")
    ).filter(col("id_filme").isNotNull())

# 3. Consolidação das 4 origens em um modelo unificado (UNION ALL)
df_cast = process_entity_column(df_bronze_credits, "cast", "Ator")
df_directors = process_entity_column(df_bronze_credits, "directors", "Diretor")
df_writers = process_entity_column(df_bronze_credits, "writers", "Roteirista")
df_companies = process_entity_column(df_bronze_credits, "production_companies", "Produtora")

df_entities_union = df_cast \
    .unionByName(df_directors) \
    .unionByName(df_writers) \
    .unionByName(df_companies)

# 4. Higienização, Capitalização e Filtro de Anomalias/Column Shift
df_entities_cleaned = df_entities_union \
    .withColumn("nome_entidade", trim(col("nome_raw"))) \
    .filter(
        (col("nome_entidade").isNotNull()) & 
        (col("nome_entidade") != "") & 
        (~col("nome_entidade").rlike("^\\d+$"))
    ) \
    .withColumn("nome_entidade", initcap(col("nome_entidade")))

# 5. Deduplicação de Registos Identicos
df_entities_dedup = df_entities_cleaned \
    .select("id_filme", "nome_entidade", "tipo_entidade") \
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"]) \
    .withColumn("processing_datetime", current_timestamp())

# 6. Gravação na Camada Silver (Delta Lake)
df_entities_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

# 7. Validação e Exibição do Resultado
print(f"[OK] Tabela 'tb_pessoas_empresas' gravada com sucesso em {silver_schema}!")
#display(spark.table(f"{silver_schema}.tb_pessoas_empresas").limit(20))

display(
    spark.table(f"{silver_schema}.tb_pessoas_empresas") # mostra contagem por tipos de entidades
    .groupBy("tipo_entidade")
    .count()
)

display(spark.table(f"{silver_schema}.tb_pessoas_empresas").limit(20)) # mostra os 20 primeiros registros

[OK] Tabela 'tb_pessoas_empresas' gravada com sucesso em workspace.cinedata_silver!


tipo_entidade,count
Produtora,120620
Roteirista,138769
Diretor,109214
Ator,548062


id_filme,nome_entidade,tipo_entidade,processing_datetime
374720,Mark Rylance,Ator,2026-09-21T12:38:24.368Z
458156,Anjelica Huston,Ator,2026-09-21T12:38:24.368Z
359940,Samara Weaving,Ator,2026-09-21T12:38:24.368Z
580489,Naomie Harris,Ator,2026-09-21T12:38:24.368Z
453395,Benedict Cumberbatch,Ator,2026-09-21T12:38:24.368Z
376867,Janelle Monáe,Ator,2026-09-21T12:38:24.368Z
328387,Brian Marc,Ator,2026-09-21T12:38:24.368Z
258489,Osy Ikhile,Ator,2026-09-21T12:38:24.368Z
497582,Burn Gorman,Ator,2026-09-21T12:38:24.368Z
296524,Dylan O'brien,Ator,2026-09-21T12:38:24.368Z



## 7. Construção da Camada Silver: `tb_cotacao_dolar`

Nesta etapa, processamos a tabela **`cinedata_bronze.tb_cotacao_dolar`** para estruturar uma série temporal diária contínua e sem lacunas.

---

### 📌 Tratamentos Aplicados:
- **Intervalo Temporal Contínuo:** Gerada uma sequência diária de datas contínuas cobrindo todo o histórico registrado na camada Bronze.
- **Técnica de Forward Fill:** Aplicação da função de janela `last(ignorenulls=True)` para preencher dias sem cotação (finais de semana e feriados) com o valor do último dia útil disponível.
- **Tipagem e Padronização:** Conversão segura dos campos para `data_cotacao` (`DATE`) e `valor_cotacao` (`DECIMAL(10,4)`).
- **Persistência:** Salvamento da série temporal completa e deduplicada na camada Silver no formato Delta Lake.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, expr, min as _min, max as _max, explode, sequence, 
    to_date, last, current_timestamp
)

# 1. Schemas do Unity Catalog
bronze_schema = "workspace.cinedata_bronze"
silver_schema = "workspace.cinedata_silver"

# 2. Leitura da Tabela Bronze
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# 3. Mapeamento inicial com o nome correto da coluna (dataHoraCotacao e cotacaoCompra)
df_cotacao_clean = df_bronze_cotacao.select(
    to_date(col("dataHoraCotacao")).alias("data_cotacao"),
    expr("try_cast(cotacaoCompra as decimal(10,4))").alias("valor_cotacao_raw")
).filter(col("data_cotacao").isNotNull()).dropDuplicates(["data_cotacao"])

# 4. Identificação do intervalo mínimo e máximo de datas
date_range = df_cotacao_clean.select(
    _min("data_cotacao").alias("min_date"),
    _max("data_cotacao").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

# 5. Criação do DataFrame com a série temporal diária contínua
df_date_grid = spark.range(1).select(
    explode(
        sequence(
            expr(f"DATE '{min_date}'"), 
            expr(f"DATE '{max_date}'"), 
            expr("INTERVAL 1 DAY")
        )
    ).alias("data_cotacao")
)

# 6. Join da grade de datas com as cotações reais e aplicação do Forward Fill
window_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_cotacao_ffill = df_date_grid \
    .join(df_cotacao_clean, on="data_cotacao", how="left") \
    .withColumn(
        "valor_cotacao",
        last("valor_cotacao_raw", ignorenulls=True).over(window_ffill)
    ) \
    .select("data_cotacao", "valor_cotacao") \
    .withColumn("processing_datetime", current_timestamp())

# 7. Gravação na Camada Silver (Delta Lake)
df_cotacao_ffill.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

# 8. Validação e Exibição do Resultado
print(f"[OK] Tabela 'tb_cotacao_dolar' gravada com sucesso em {silver_schema}!")
display(spark.table(f"{silver_schema}.tb_cotacao_dolar").orderBy("data_cotacao").limit(15))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela 'tb_cotacao_dolar' gravada com sucesso em workspace.cinedata_silver!


data_cotacao,valor_cotacao,processing_datetime
2016-09-21,3.2402,2026-09-21T12:38:33.846Z
2016-09-22,3.2009,2026-09-21T12:38:33.846Z
2016-09-23,3.2236,2026-09-21T12:38:33.846Z
2016-09-24,3.2236,2026-09-21T12:38:33.846Z
2016-09-25,3.2236,2026-09-21T12:38:33.846Z
2016-09-26,3.2394,2026-09-21T12:38:33.846Z
2016-09-27,3.2352,2026-09-21T12:38:33.846Z
2016-09-28,3.2470,2026-09-21T12:38:33.846Z
2016-09-29,3.2229,2026-09-21T12:38:33.846Z
2016-09-30,3.2456,2026-09-21T12:38:33.846Z


In [0]:
display(
    spark.table("workspace.cinedata_bronze.tb_cotacao_dolar")
    .select("dataHoraCotacao", "cotacaoCompra")
)

dataHoraCotacao,cotacaoCompra
2016-09-21 00:00:00,3.2402
2016-09-22 00:00:00,3.2009
2016-09-23 00:00:00,3.2236
2016-09-26 00:00:00,3.2394
2016-09-27 00:00:00,3.2352
2016-09-28 00:00:00,3.247
2016-09-29 00:00:00,3.2229
2016-09-30 00:00:00,3.2456
2016-10-03 00:00:00,3.2332
2016-10-04 00:00:00,3.2197
